In [1]:
# Cell 1: imports

from functools import partial
import importlib

import matplotlib.pyplot as plt
import torch

import vt_all_solvers_wrapper as vt
import dataset
import utils
import models.mlp as mlp_module
import models.fno as fno_module
import models.deep as deep_module

importlib.reload(vt)
importlib.reload(dataset)
importlib.reload(utils)
importlib.reload(mlp_module)
importlib.reload(fno_module)
importlib.reload(deep_module)

from base_range_library import range_library
from dataset import StreamingGeometryDatasetConfig, make_streaming_dataloader
from utils import (
    DEFAULT_VALIDATION_METRICS,
    TrainConfig,
    fit,
    make_optimizer,
    mse_loss,
    plot_history,
)
from losses import UniversalTransferFunctionLoss
from models.mlp import ProfileMLP, webster_mlp_batch_to_xy
from models.fno import TransferFunctionFNO, webster_fno_batch_to_xy
from models.deep import TransferFunctionDeepONet, webster_deeponet_batch_to_xy


In [ ]:
# Cell 2: common config

device = "cuda" if torch.cuda.is_available() else "cpu"

batch_size = 32
sections = 100
n_profile_points = 128
n_frequencies = 256

steps_per_epoch = 10
val_steps = 10
epochs = 20

# Change this to inspect another channel inside the validation batch.
preview_sample_idx = 0

mlp_checkpoint_name = "mlp_db"
fno_checkpoint_name = "fno_db"
deeponet_checkpoint_name = "deeponet_db"

loss_name = "transfer_db"  # "mse" or "transfer_db"
transfer_loss_db_kwargs = {
    "output_type": "db",
    "db_weight": 1.0,
    "magnitude_weight": 0.1,
    "db_derivative_weight": 0.02,
    "peak_weight": 0.0,
    "complex_weight": 0.0,
    "complex_derivative_weight": 0.0,
    "phase_weight": 0.0,
    "db_error_scale": 10.0,
    "db_derivative_scale": 0.01,
    "peak_temperature_db": 3.0,
    "phase_dynamic_range_db": 40.0,
    "min_db": -100.0,
    "eps": 1e-8,
}

solver_config = vt.SolverConfig(
    solver="cone",
    sections=sections,
    points=n_frequencies,
    f_min_hz=50.0,
    f_max_hz=5000.0,
    grid="linear",
)

train_config = StreamingGeometryDatasetConfig(
    geometry_kind="random",
    solver_config=solver_config,
    target_mode="db",
    seed=1,
)

val_config = StreamingGeometryDatasetConfig(
    geometry_kind="random",
    solver_config=solver_config,
    target_mode="db",
    seed=100_000,
    max_samples=batch_size * val_steps,
)

print("device:", device)


In [ ]:
# Cell 3: dataloaders

train_loader = make_streaming_dataloader(
    train_config,
    range_library,
    batch_size=batch_size,
    num_workers=0,
)

val_loader = make_streaming_dataloader(
    val_config,
    range_library,
    batch_size=batch_size,
    num_workers=0,
)


In [ ]:
# Cell 4: batch adapters

mlp_batch_to_xy = partial(
    webster_mlp_batch_to_xy,
    n_points=n_profile_points,
    log_area=True,
    include_x=True,
)

fno_batch_to_xy = partial(
    webster_fno_batch_to_xy,
    n_points=n_profile_points,
    log_area=True,
)

deeponet_batch_to_xy = partial(
    webster_deeponet_batch_to_xy,
    n_points=n_profile_points,
    log_area=True,
    frequency_min_hz=solver_config.f_min_hz,
    frequency_max_hz=solver_config.f_max_hz,
)


In [ ]:
# Cell 5: validation metrics and loss

validation_metrics = list(DEFAULT_VALIDATION_METRICS.keys())

transfer_loss_db = UniversalTransferFunctionLoss(**transfer_loss_db_kwargs)

loss_registry = {
    "mse": mse_loss,
    "transfer_db": transfer_loss_db,
}

criterion = loss_registry[loss_name]

print("loss:", criterion)
print("validation metrics:", validation_metrics)


In [ ]:
# Cell 6: preview helpers

def call_model(model, inputs):
    if isinstance(inputs, tuple):
        return model(*inputs)
    if isinstance(inputs, list):
        return model(*inputs)
    if isinstance(inputs, dict):
        return model(**inputs)
    return model(inputs)


def predict_on_batch(model, batch_to_xy, batch):
    model.eval()
    with torch.no_grad():
        inputs, target = batch_to_xy(batch, torch.device(device))
        prediction = call_model(model, inputs)
    return prediction.detach().cpu(), target.detach().cpu()


def validation_preview_batch():
    return next(iter(val_loader))


def plot_single_model_preview(model, batch_to_xy, model_label, *, sample_idx=preview_sample_idx):
    batch = validation_preview_batch()
    prediction, target = predict_on_batch(model, batch_to_xy, batch)

    if sample_idx < 0 or sample_idx >= target.shape[0]:
        raise IndexError(
            f"sample_idx={sample_idx} is outside validation batch size {target.shape[0]}"
        )

    freq = batch["frequencies_hz"][sample_idx].detach().cpu()
    pred = prediction[sample_idx]
    y = target[sample_idx]

    plt.figure(figsize=(11, 4))
    plt.plot(freq, y, label="target dB", linewidth=2.0)
    plt.plot(freq, pred, label=model_label, linewidth=1.8)
    plt.xlabel("Frequency, Hz")
    plt.ylabel("Transfer function, dB")
    plt.title(f"{model_label}: validation sample {sample_idx}")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    mae = torch.mean(torch.abs(pred - y)).item()
    rmse = torch.sqrt(torch.mean((pred - y) ** 2)).item()
    print(f"{model_label} preview MAE dB:  {mae:.4f}")
    print(f"{model_label} preview RMSE dB: {rmse:.4f}")


### MLP

In [ ]:
# Cell 7: train MLP

mlp = ProfileMLP(
    n_profile_points=n_profile_points,
    in_channels=2,  # log(area), x
    n_frequencies=n_frequencies,
    hidden_dim=512,
    depth=4,
    dropout=0.05,
    out_channels=1,
).to(device)

mlp_optimizer = make_optimizer(
    mlp,
    lr=1e-3,
    weight_decay=1e-4,
)

mlp_history = fit(
    mlp,
    mlp_optimizer,
    train_loader,
    val_loader,
    criterion=criterion,
    batch_to_xy=mlp_batch_to_xy,
    config=TrainConfig(
        epochs=epochs,
        steps_per_epoch=steps_per_epoch,
        val_steps=val_steps,
        device=device,
        show_progress=False,
        live_plot_every_steps=50,
        grad_clip_norm=None,
        checkpoint_name=mlp_checkpoint_name,
        save_every_steps=None,
        validation_metrics=validation_metrics,
    ),
)

plot_history(mlp_history)
plot_single_model_preview(mlp, mlp_batch_to_xy, "MLP")


### FNO

In [ ]:
# Cell 8: train FNO

fno = TransferFunctionFNO(
    n_modes=32,
    hidden_channels=96,
    latent_dim=256,
    pooling_bins=16,
    frequency_bands=16,
    out_channels=1,
).to(device)

fno_optimizer = make_optimizer(
    fno,
    lr=5e-4,
    weight_decay=1e-4,
)

fno_history = fit(
    fno,
    fno_optimizer,
    train_loader,
    val_loader,
    criterion=criterion,
    batch_to_xy=fno_batch_to_xy,
    config=TrainConfig(
        epochs=epochs,
        steps_per_epoch=steps_per_epoch,
        val_steps=val_steps,
        device=device,
        show_progress=False,
        live_plot_every_steps=50,
        grad_clip_norm=None,
        checkpoint_name=fno_checkpoint_name,
        save_every_steps=None,
        validation_metrics=validation_metrics,
    ),
)

plot_history(fno_history)
plot_single_model_preview(fno, fno_batch_to_xy, "FNO")


### DeepONet

In [ ]:
# Cell 9: train DeepONet

deeponet = TransferFunctionDeepONet(
    in_channels=1,
    hidden_channels=64,
    basis_dim=128,
    pooling_bins=16,
    frequency_bands=16,
    trunk_hidden_dim=128,
    out_channels=1,
).to(device)

deeponet_optimizer = make_optimizer(
    deeponet,
    lr=5e-4,
    weight_decay=1e-4,
)

deeponet_history = fit(
    deeponet,
    deeponet_optimizer,
    train_loader,
    val_loader,
    criterion=criterion,
    batch_to_xy=deeponet_batch_to_xy,
    config=TrainConfig(
        epochs=epochs,
        steps_per_epoch=steps_per_epoch,
        val_steps=val_steps,
        device=device,
        show_progress=False,
        live_plot_every_steps=50,
        grad_clip_norm=None,
        checkpoint_name=deeponet_checkpoint_name,
        save_every_steps=None,
        validation_metrics=validation_metrics,
    ),
)

plot_history(deeponet_history)
plot_single_model_preview(deeponet, deeponet_batch_to_xy, "DeepONet")


### Сравнение

In [ ]:
# Cell 10: compare all models on one validation channel

comparison_sample_idx = preview_sample_idx
comparison_batch = validation_preview_batch()

models_to_compare = [
    ("MLP", mlp, mlp_batch_to_xy),
    ("FNO", fno, fno_batch_to_xy),
    ("DeepONet", deeponet, deeponet_batch_to_xy),
]

predictions = {}
target = None

for name, model, batch_to_xy in models_to_compare:
    prediction, current_target = predict_on_batch(model, batch_to_xy, comparison_batch)
    predictions[name] = prediction
    target = current_target if target is None else target

if comparison_sample_idx < 0 or comparison_sample_idx >= target.shape[0]:
    raise IndexError(
        f"comparison_sample_idx={comparison_sample_idx} is outside validation batch size {target.shape[0]}"
    )

freq = comparison_batch["frequencies_hz"][comparison_sample_idx].detach().cpu()
y = target[comparison_sample_idx]

plt.figure(figsize=(12, 5))
plt.plot(freq, y, label="target dB", linewidth=2.4, color="black")

for name, prediction in predictions.items():
    plt.plot(freq, prediction[comparison_sample_idx], label=name, linewidth=1.8, alpha=0.85)

plt.xlabel("Frequency, Hz")
plt.ylabel("Transfer function, dB")
plt.title(f"Model comparison on validation sample {comparison_sample_idx}")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print("=== Validation sample metrics ===")
for name, prediction in predictions.items():
    pred = prediction[comparison_sample_idx]
    mae = torch.mean(torch.abs(pred - y)).item()
    rmse = torch.sqrt(torch.mean((pred - y) ** 2)).item()
    print(f"{name:8s} MAE dB: {mae:8.4f} | RMSE dB: {rmse:8.4f}")


In [ ]:
# Cell 11: compare training histories

histories = {
    "MLP": mlp_history,
    "FNO": fno_history,
    "DeepONet": deeponet_history,
}

plt.figure(figsize=(10, 4))

for name, history in histories.items():
    if history.val_loss:
        plt.plot(history.val_loss, marker="o", label=f"{name} val")
    if history.train_loss:
        plt.plot(history.train_loss, linestyle="--", alpha=0.65, label=f"{name} train")

plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Training history comparison")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print("=== Best validation loss ===")
for name, history in histories.items():
    best_val = min(history.val_loss) if history.val_loss else None
    print(f"{name:8s} best val loss: {best_val}")
